# Deepchecks Tabular — Model Evaluation (Checks seleccionados)

Este notebook (Colab) demuestra el uso de los siguientes checks de **Deepchecks Tabular**:

- **Model Error Analysis** — patrones de error
- **Single Feature Contribution** — contribución/impacto por feature
- **Unused Features** — variables sin impacto en el modelo
- **Boosting Overfit** — sobreajuste en modelos boosting/ensemble
- **Performance Report** — métricas globales
- **Confusion Matrix Report** — errores por clase
- **ROC / PR Analysis** — curvas ROC y Precision-Recall

> Dataset de ejemplo: `breast_cancer` (scikit-learn). Modelo principal: `GradientBoostingClassifier`.


# Contexto de métricas e informes de análisis de modelos

Todo esto suele aparecer en herramientas de **análisis y validación de modelos** y sirve para responder una pregunta central:

> **“Mi modelo funciona, pero exactamente dónde acierta, dónde falla y por qué”.**

---

## 1. Model Error Analysis — patrones de error

Aquí no miras solo una métrica global, sino **qué tipo de ejemplos está fallando el modelo**.

Sirve para detectar cosas como:

- falla más en una clase concreta
- se equivoca en ciertos rangos de una variable
- comete errores en casos raros o con ruido
- rinde peor en ciertos segmentos del dataset

### Ejemplo

Un modelo de fraude puede tener buen AUC general, pero fallar mucho en:

- transacciones pequeñas
- ciertos países
- clientes nuevos

La idea es pasar de algo como:

> “El modelo tiene 0.87 AUC”

a algo más útil como:

> “El modelo se confunde sobre todo en observaciones borderline y en clientes con poco histórico”.

### ¿Para qué sirve?

Ayuda a decidir si necesitas:

- más datos
- nuevas features
- rebalanceo
- segmentar modelos

---

## 2. Single Feature Contribution — contribución/impacto por feature

Esto intenta responder:

> **“¿Cómo está influyendo cada variable en la predicción?”**

Puede verse de varias formas:

- importancia global de variables
- impacto local por observación
- dirección del efecto: subir una variable aumenta o reduce la probabilidad predicha

Aquí suelen entrar ideas como:

- feature importance
- permutation importance
- SHAP values
- PDP / ICE plots

### Ejemplo

Si `volumen_relativo` tiene alta contribución positiva cuando sube, podrías interpretar:

> “Cuando el volumen relativo es alto, el modelo tiende a predecir subida con más confianza”.

### Importante

- **contribución no implica causalidad**
- en modelos con variables correlacionadas, la importancia puede repartirse o distorsionarse

---

## 3. Unused Features — variables sin impacto en el modelo

Son variables que, en la práctica, **el modelo casi no está usando**.

Puede pasar porque:

- no tienen señal
- están duplicadas o muy correlacionadas con otras
- su varianza es muy baja
- el algoritmo no encuentra splits o pesos útiles para ellas

### ¿Para qué sirve?

Sirve para:

- simplificar el modelo
- reducir ruido
- mejorar interpretabilidad
- abaratar inferencia

### Ejemplo

Si tienes 80 variables y 20 no aportan nada, puedes quedarte con una versión más limpia sin perder rendimiento.

### Ojo

Que una feature salga como “unused” no siempre significa que sea inútil en absoluto. A veces:

- otra variable equivalente ya captura esa información
- solo sería útil en otro modelo
- solo ayuda en un subconjunto pequeño de casos

---

## 4. Boosting Overfit — sobreajuste en modelos boosting/ensemble

Esto se refiere a detectar si un modelo tipo:

- XGBoost
- LightGBM
- CatBoost
- Gradient Boosting

está aprendiendo demasiado el training y luego generaliza mal.

### Se ve cuando

- la métrica en train es muy buena
- la de validación empeora o se estanca
- al añadir más árboles el modelo sigue mejorando en train pero no fuera de muestra

### Señales típicas

- gap grande entre train y validation
- profundidad excesiva
- demasiados estimadores
- learning rate y complejidad mal ajustados

### Ejemplo

- Train AUC = 0.99
- Validation AUC = 0.74

Eso suele indicar que el modelo está memorizando patrones del train.

### Qué revisar

- early stopping
- `max_depth`
- `min_child_weight` / `min_data_in_leaf`
- subsampling
- regularización
- leakage de variables

---

## 5. Performance Report — métricas globales

Es el resumen de rendimiento general del modelo.

### En clasificación puede incluir

- accuracy
- precision
- recall
- F1
- AUC-ROC
- log loss
- MCC
- balanced accuracy

### En regresión puede incluir

- MAE
- RMSE
- R²
- MAPE

Sirve para una visión rápida de:

> **“Qué tan bueno es el modelo en conjunto”.**

### Limitación importante

Una métrica global **esconde dónde falla**.

### Ejemplo

Un 92% de accuracy puede parecer excelente, pero si el dataset está desbalanceado quizá el modelo casi nunca detecta la clase minoritaria.

Por eso este informe suele ser el punto de partida, no el final del análisis.

---

## 6. Confusion Matrix Report — errores por clase

La matriz de confusión descompone las predicciones en:

- verdaderos positivos
- falsos positivos
- verdaderos negativos
- falsos negativos

Es muy útil para entender **qué tipo de error estás cometiendo**.

### En binario

- **FP**: predice positivo cuando no lo era
- **FN**: no detecta un positivo real

### Ejemplo médico

- FN puede ser más grave que FP, porque dejas pasar una enfermedad real

### Ejemplo fraude

- demasiados FP molestan a clientes legítimos
- demasiados FN dejan escapar fraude

### En multiclase

Además ves **entre qué clases se confunde** el modelo.

Por ejemplo:

> “La clase B se predice muchas veces como A”.

Eso da pistas para mejorar features o incluso redefinir etiquetas.

---

## 7. ROC / PR Analysis — curvas ROC y Precision-Recall

Ambas evalúan clasificadores binarios en distintos umbrales.

### ROC curve

Relaciona:

- TPR = recall / sensibilidad
- FPR = tasa de falsos positivos

El **AUC-ROC** mide capacidad de separar clases en general.

Es útil cuando quieres ver la capacidad discriminativa total del modelo.

### PR curve

Relaciona:

- precision
- recall

Es especialmente útil cuando la clase positiva es rara o el dataset está desbalanceado.

### Ejemplo

En fraude o detección de eventos raros, la curva PR suele ser más informativa que ROC.

### ¿Por qué?

Porque ROC puede parecer buena incluso cuando el modelo genera demasiados falsos positivos en términos prácticos.

### Regla útil

- **ROC**: visión general de separabilidad
- **PR**: mejor para clases minoritarias y problemas donde importa mucho la calidad de los positivos detectados

---

## Cómo se conectan entre sí

Una forma útil de verlo es así:

- **Performance Report**: qué tan bien va en general
- **Confusion Matrix**: en qué tipo de errores cae
- **ROC / PR**: cómo cambia el rendimiento según el umbral
- **Model Error Analysis**: en qué segmentos o casos se equivoca
- **Single Feature Contribution**: qué variables empujan las predicciones
- **Unused Features**: qué variables no están aportando
- **Boosting Overfit**: si el modelo está aprendiendo demasiado el train

---

## Ejemplo práctico de lectura

Imagina un clasificador de subida/bajada en trading:

- **Performance Report**: AUC 0.74, F1 0.58
- **Confusion Matrix**: muchos falsos positivos en señales BUY
- **ROC/PR**: AUC razonable, pero PR baja en la clase positiva
- **Error Analysis**: falla más en periodos de baja volatilidad
- **Single Feature Contribution**: volumen relativo y eficiencia explican gran parte de la señal
- **Unused Features**: RSI lento y algunas medias redundantes no aportan
- **Boosting Overfit**: train AUC 0.93 vs valid AUC 0.74, señal clara de sobreajuste

### Conclusión

> El modelo sí detecta señal, pero se vuelve demasiado optimista fuera de muestra y se equivoca sobre todo en mercados laterales.

Eso ya te dice qué tocar después.

---

## Resumen en una frase

- **Model Error Analysis**: dónde y cómo falla el modelo
- **Single Feature Contribution**: qué aporta cada variable a la predicción
- **Unused Features**: qué variables no están ayudando
- **Boosting Overfit**: si el ensemble está memorizando en vez de generalizar
- **Performance Report**: resumen global de métricas
- **Confusion Matrix Report**: desglose de errores por clase
- **ROC / PR Analysis**: comportamiento del clasificador al cambiar el umbral

In [1]:
!pip install "deepchecks" -q


import numpy as np
np.Inf = np.inf
np.NINF = -np.inf
np.PINF = np.inf
np.NaN = np.nan


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
#@title 2) Imports
import numpy as np
import pandas as pd

# CAMBIO: Usamos load_wine en lugar de load_breast_cancer
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression

from deepchecks.tabular import Dataset, Suite
from deepchecks.tabular.suites import model_evaluation

c:\Users\tteru\.pyenv\pyenv-win\versions\3.12.8\Lib\site-packages\deepchecks\core\serialization\dataframe\html.py:16: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version


In [3]:
from deepchecks.tabular.checks import (
    TrainTestPerformance,
    WeakSegmentsPerformance,
    UnusedFeatures,
    BoostingOverfit,
    ConfusionMatrixReport,
    RocReport
)

In [4]:
#@title 3) Cargar datos y preparar Dataset (Deepchecks)

# CAMBIO: Cargamos los datos de vinos
data = load_wine(as_frame=True)
df = data.frame.copy()

target_col = 'target'
X = df.drop(columns=[target_col])
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

train_df = X_train.copy()
train_df[target_col] = y_train.values

test_df = X_test.copy()
test_df[target_col] = y_test.values

train_ds = Dataset(train_df, label=target_col)
test_ds  = Dataset(test_df,  label=target_col)

print('Train:', train_df.shape, 'Test:', test_df.shape)
train_df.head()

deepchecks - WARNING - It is recommended to initialize Dataset with categorical features by doing "Dataset(df, cat_features=categorical_list)". No categorical features were passed, therefore heuristically inferring categorical features in the data. 0 categorical features were inferred.
deepchecks - WARNING - It is recommended to initialize Dataset with categorical features by doing "Dataset(df, cat_features=categorical_list)". No categorical features were passed, therefore heuristically inferring categorical features in the data. 0 categorical features were inferred.


Train: (133, 14) Test: (45, 14)


,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline,target
8,14.83,1.64,2.17,14.0,97.0,2.8,2.98,0.29,1.98,5.20,1.08,2.85,1045.0,0
104,12.51,1.73,1.98,20.5,85.0,2.2,1.92,0.32,1.48,2.94,1.04,3.57,672.0,1
36,13.28,1.64,2.84,15.5,110.0,2.6,2.68,0.34,1.36,4.60,1.09,2.78,880.0,0
78,12.33,0.99,1.95,14.8,136.0,1.9,1.85,0.35,2.76,3.40,1.06,2.31,750.0,1
2,13.16,2.36,2.67,18.6,101.0,2.8,3.24,0.30,2.81,5.68,1.03,3.17,1185.0,0


In [5]:
#@title 4) Entrenar modelos (Boosting + baseline opcional)
# Modelo Boosting (para BoostingOverfit)
gb = GradientBoostingClassifier(random_state=42)
gb.fit(X_train, y_train)

# Baseline opcional
lr = LogisticRegression(max_iter=2000)
lr.fit(X_train, y_train)

print("GradientBoosting train score:", gb.score(X_train, y_train))
print("GradientBoosting test score :", gb.score(X_test, y_test))
print("LogReg          train score:", lr.score(X_train, y_train))
print("LogReg          test score :", lr.score(X_test, y_test))

GradientBoosting train score: 1.0
GradientBoosting test score : 0.9777777777777777
LogReg          train score: 0.9924812030075187
LogReg          test score : 0.9555555555555556


c:\Users\tteru\.pyenv\pyenv-win\versions\3.12.8\Lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [6]:
#@title 5) Validación rápida (classification_report)
pred = gb.predict(X_test)
print(classification_report(y_test, pred, digits=4))

              precision    recall  f1-score   support

           0     0.9375    1.0000    0.9677        15
           1     1.0000    0.9444    0.9714        18
           2     1.0000    1.0000    1.0000        12

    accuracy                         0.9778        45
   macro avg     0.9792    0.9815    0.9797        45
weighted avg     0.9792    0.9778    0.9778        45



## Opción A — Suite completa de Model Evaluation

Deepchecks incluye una suite estándar para evaluación del modelo. Puedes ejecutarla y explorar el resultado interactivo.


In [7]:
#@title 6A) Ejecutar suite estándar
suite_std = model_evaluation()
result_std = suite_std.run(train_dataset=train_ds, test_dataset=test_ds, model=gb)
result_std

Accordion(children=(VBox(children=(HTML(value='\n<h1 id="summary_NP08VHOFH1ZTYV907G2LNG0VZ">Model Evaluation S…

## Opción B — Suite custom (solo checks seleccionados)

Aquí ejecutamos exactamente los checks solicitados.


## Ejecutar checks uno a uno (útil para docencia)

Esta sección imprime cada check por separado.


In [8]:


# Matriz de confusión
result = ConfusionMatrixReport().run(test_ds, gb)
result.show()

# ROC
result = RocReport().run(test_ds, gb)
result.show()

# Segmentos débiles
result = WeakSegmentsPerformance().run(test_ds, gb)
result.show()

# Variables no usadas
result = UnusedFeatures().run(test_ds, gb)
result.show()

## Exportar resultados a HTML

Genera un informe interactivo para compartir (Confluence/SharePoint) o adjuntar.


In [9]:
# Guardar el reporte completo de la suite "model_evaluation" en HTML
output_file = "deepchecks_ML_eval_wine.html"

# result_std es la variable de tu celda 7 donde corriste la suite completa
result_std.save_as_html(output_file)

print(f"✅ Reporte completo de evaluación guardado con éxito como: {output_file}")
print("¡Importante! Ve a la carpeta donde está este notebook, busca el archivo HTML, comprímelo en un .zip y súbelo a la plataforma.")

✅ Reporte completo de evaluación guardado con éxito como: deepchecks_ML_eval_wine.html
¡Importante! Ve a la carpeta donde está este notebook, busca el archivo HTML, comprímelo en un .zip y súbelo a la plataforma.


---
### Notas
- Si cambias el modelo a XGBoost/LightGBM, algunos checks como **BoostingOverfit** pueden ser todavía más informativos.
- Si tu problema es multiclase, la matriz de confusión y el análisis de errores se vuelve más rico.
